In [ ]:
import os 
import subprocess 

from datasets import load_dataset
from dotenv import load_dotenv 
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
import matplotlib.pyplot as plt 
import numpy as np
from matplotlib.lines import Line2D

from flower.evaluation.metrics import evaluate_embedding_classifier, evaluate_embedding_regressor, prepare_data

load_dotenv()
DATA_ROOT = os.getenv("DATA_ROOT")

In [4]:
betas = [0., 2.0, 5.0, 10., 50., 100.]

In [13]:
RANDOM_STATE = 42

clf_models = {
    'log_regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    '2-mlp': MLPClassifier(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
}

reg_models = {
    'lin_regression': LinearRegression(),
    '2-mlp': MLPRegressor(
        hidden_layer_sizes=(64, 32), 
        max_iter=1000, 
        random_state=42
    )
}

In [ ]:
# Note: REPLACE experiment name and model number in this block

import pickle
import json

all_results = []
for i in range(6):
    data_files = {
            "train": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior_beta_ablation/embeddings/7578763_{i}/train/*.parquet",
            "test": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior_beta_ablation/embeddings/7578763_{i}/test/*.parquet"
        }
    ds = load_dataset(
        "parquet",
        data_files=data_files
    )
    X_train, y_train, X_test, y_test = prepare_data(
        ds=ds,
        embed_type="cond",
        factor="digit",
    )
   
    iteration_data = {
        "iteration": i,
        "classification": [],
        "regression": []
    }

    for model_name, clf_model in clf_models.items():
        print(f"Running Iteration {i}, Model: {model_name}")
        res = evaluate_embedding_classifier(
            X_train=X_train, X_test=X_test,
            y_train=y_train, y_test=y_test,
            model=clf_model
        )
        # Add metadata to the returned dictionary
        res["model_name"] = model_name
        iteration_data["classification"].append(res)
    
    
    for factor in ["r", "g", "b"]:
        _, y_train, _, y_test = prepare_data(
            ds=ds,
            embed_type="orig",
            factor=factor,
        )

        for model_name, reg_model in reg_models.items():
            res = evaluate_embedding_regressor(
                X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test,
                model=reg_model
            )
            res["model_name"] = model_name
            res["factor"] = factor
            iteration_data["regression"].append(res)


    all_results.append(iteration_data)

file_path = "model_results.pkl"
with open(file_path, "wb") as f:
    pickle.dump(all_results, f)

print(f"Successfully saved results for {len(all_results)} iterations to {file_path}")

In [ ]:
import pickle

file_path = "./results/model_results.pkl"

# Open the file in 'rb' (read binary) mode
with open(file_path, "rb") as f:
    all_results = pickle.load(f) 

print(f"Successfully loaded {len(all_results)} iterations.")

In [ ]:
# ... (Keep your data extraction and beta=1.0 manual insertion code here) ...
# 1. Setup Beta mapping
betas = [0.0, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0]

# Initialize data structures
class_plot_data = {'Linear Probe': {'m': [], 'l': [], 'h': []}, 'MLP': {'m': [], 'l': [], 'h': []}}
reg_plot_data = {
    f: {'Linear Probe': {'m': [], 'l': [], 'h': []}, 'MLP': {'m': [], 'l': [], 'h': []}} 
    for f in ['R', 'G', 'B']
}

# 2. Extract Data from all_results (assuming it contains betas 0, 2, 5, 10, 50, 100)
for i, iteration in enumerate(all_results):
    # Classification
    for res in iteration['classification']:
        name = 'Linear Probe' if res['model_name'] == 'log_regression' else 'MLP'
        stats = res['bootstrap_accuracy']
        class_plot_data[name]['m'].append(stats['median'])
        class_plot_data[name]['l'].append(stats['ci_95'][0])
        class_plot_data[name]['h'].append(stats['ci_95'][1])
        
    # Regression
    for res in iteration['regression']:
        name = 'Linear Probe' if res['model_name'] == 'lin_regression' else 'MLP'
        f = res['factor'].upper()
        stats = res['bootstrap']
        reg_plot_data[f][name]['m'].append(stats['median'])
        reg_plot_data[f][name]['l'].append(stats['ci_95'][0])
        reg_plot_data[f][name]['h'].append(stats['ci_95'][1])

# 3. Manually insert the Beta = 1.0 results at index 1 from the benchmark.ipynb notebook.
# Classification beta=1.0
class_plot_data['Linear Probe']['m'].insert(1, 0.7486)
class_plot_data['Linear Probe']['l'].insert(1, 0.7486 - 0.0083)
class_plot_data['Linear Probe']['h'].insert(1, 0.7486 + 0.0083)

class_plot_data['MLP']['m'].insert(1, 0.7888)
class_plot_data['MLP']['l'].insert(1, 0.7888 - 0.0081)
class_plot_data['MLP']['h'].insert(1, 0.7888 + 0.0081)

# Regression beta=1.0 (R, G, B)
# Red
reg_plot_data['R']['Linear Probe']['m'].insert(1, 0.4930); reg_plot_data['R']['Linear Probe']['l'].insert(1, 0.4930 - 0.0125); reg_plot_data['R']['Linear Probe']['h'].insert(1, 0.4930 + 0.0125)
reg_plot_data['R']['MLP']['m'].insert(1, 0.5161); reg_plot_data['R']['MLP']['l'].insert(1, 0.5161 - 0.0152); reg_plot_data['R']['MLP']['h'].insert(1, 0.5161 + 0.0152)
# Green
reg_plot_data['G']['Linear Probe']['m'].insert(1, 0.5394); reg_plot_data['G']['Linear Probe']['l'].insert(1, 0.5394 - 0.0126); reg_plot_data['G']['Linear Probe']['h'].insert(1, 0.5394 + 0.0126)
reg_plot_data['G']['MLP']['m'].insert(1, 0.6343); reg_plot_data['G']['MLP']['l'].insert(1, 0.6343 - 0.0133); reg_plot_data['G']['MLP']['h'].insert(1, 0.6343 + 0.0133)
# Blue
reg_plot_data['B']['Linear Probe']['m'].insert(1, 0.9225); reg_plot_data['B']['Linear Probe']['l'].insert(1, 0.9225 - 0.0026); reg_plot_data['B']['Linear Probe']['h'].insert(1, 0.9225 + 0.0026)
reg_plot_data['B']['MLP']['m'].insert(1, 0.9584); reg_plot_data['B']['MLP']['l'].insert(1, 0.9584 - 0.0017); reg_plot_data['B']['MLP']['h'].insert(1, 0.9584 + 0.0017)


# Create a figure with two subplots side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), dpi=300)

# --- SUBPLOT 1: CLASSIFICATION ---
model_colors = {'Linear Probe': '#555555', 'MLP': '#673AB7'}

for model in ['Linear Probe', 'MLP']:
    m = np.array(class_plot_data[model]['m'])
    yerr = [m - np.array(class_plot_data[model]['l']), np.array(class_plot_data[model]['h']) - m]
    
    # Set dashed for Linear Probe, solid for MLP
    ls = '--' if model == 'Linear Probe' else '-'
    
    ax1.errorbar(betas, m, yerr=yerr, label=model, color=model_colors[model], 
                 linestyle=ls, marker='o', capsize=5, markersize=7, linewidth=2)

ax1.set_xlabel('$\\beta$', fontsize=30)
ax1.set_ylabel('Accuracy', fontsize=24)
ax1.set_xscale('symlog', linthresh=1)
ax1.set_title('Digit classification', fontsize=24, pad=15)
ax1.legend(loc='center right', fontsize=20)

# --- SUBPLOT 2: REGRESSION ---
factor_colors = {'R': '#D32F2F', 'G': '#388E3C', 'B': '#1976D2'}

for f in ['R', 'G', 'B']:
    for model in ['Linear Probe', 'MLP']:
        m = np.array(reg_plot_data[f][model]['m'])
        yerr = [m - np.array(reg_plot_data[f][model]['l']), np.array(reg_plot_data[f][model]['h']) - m]
        
        linestyle = '-' if model == 'MLP' else '--'
        marker = 's' if model == 'MLP' else '^'
        
        ax2.errorbar(betas, m, yerr=yerr, color=factor_colors[f], 
                     linestyle=linestyle, marker=marker, 
                     capsize=3, markersize=6, alpha=0.8)

reg_legend_elements = [
    Line2D([0], [0], color='black', lw=2, label='MLP (Solid)'),
    Line2D([0], [0], color='black', lw=2, ls='--', label='Linear Probe (Dashed)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#D32F2F', markersize=10, label='Red Factor'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#388E3C', markersize=10, label='Green Factor'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#1976D2', markersize=10, label='Blue Factor')
]

ax2.set_xlabel('$\\beta$', fontsize=30)
ax2.set_ylabel('$R^2$ Score', fontsize=30)
ax2.set_xscale('symlog', linthresh=1)
ax2.set_title(r'$r$, $g$, $b$ regression', fontsize=24, pad=15)
ax2.legend(handles=reg_legend_elements, loc='center right', fontsize=20, ncol=1)

plt.tight_layout()
#plt.savefig("./acc_r2_beta_ablation.pdf")
plt.show()

## Beta and Losses

In [ ]:
import os

from dotenv import load_dotenv
import wandb
import matplotlib.pyplot as plt 
import scienceplots 

load_dotenv()

name = os.getenv("WANDB_ENTITY")
project = os.getenv("WANDB_PROJECT")
DATA_ROOT = os.getenv("DATA_ROOT")

api = wandb.Api()

In [18]:
betas = [0., 2.0, 5.0, 10., 50., 100.]

In [14]:
kl_losses = []
cfm_losses = []
losses = []

for i in range(6):
    run = api.run(f"/{name}/{project}/runs/7578763_{i}")
    x = run.history(samples=10000)
    cfm_losses.append(x["test_cfm_loss"].dropna())
    losses.append(x["test_loss"].dropna())
    kl_losses.append(x["test_kl_loss"].dropna())

In [ ]:
# 1. Setup the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

# Colorblind-friendly hex codes (Okabe-Ito Palette)
# Using Orange for KL and Reddish Purple for CFM
color_1 = '#E69F00' # Orange
color_2 = '#CC79A7' # Reddish Purple
grid_style = dict(ls='-', alpha=0.2)

# --- SUBPLOT 1: KL LOSS ---
ax1.plot(betas, kl_losses, color=color_1, marker='o', markersize=8, 
         linewidth=2.5, alpha=0.9, label='KL Divergence')

ax1.set_yscale('log')
ax1.set_xlabel(r'$\beta$', fontsize=24)
ax1.set_ylabel(r'$\text{KL} \left[ p(x_0|y)\| q(x_0) \right]$', fontsize=24)
ax1.grid(True, which="both", **grid_style)

# --- SUBPLOT 2: CFM LOSS ---
ax2.plot(betas, cfm_losses, color=color_2, marker='s', markersize=8, 
         linewidth=2.5, alpha=0.9, label='CFM Loss')

ax2.set_yscale('log')
ax2.set_xlabel(r'$\beta$', fontsize=24)
ax2.set_ylabel(r'$\mathcal{L}_{\text{CFM}}$', fontsize=24)
ax2.grid(True, which="both", **grid_style)

# Clean up layout
plt.tight_layout()
plt.savefig("./losses_beta_ablation.pdf")
plt.show()